# 02 — Exact model and initialization

## Goal

Implement and verify only the NoisyMNIST network architecture and initialization needed
for the faithful reproduction. This phase creates no optimizer and performs no training.

Phase 1 remains authoritative for dataset construction, splits, and online streams:
[01_noisymnist_dataset_and_protocol.ipynb](01_noisymnist_dataset_and_protocol.ipynb).


## Source audit

The primary source available for this project is the official Oak Lab research note,
[Learning from experience instead of curated datasets](https://www.oaklab.ai/posts/learning-from-experience-instead-of-curated-datasets),
published July 13, 2026. No PDF, supplementary material, source repository, or locally
stored implementation was found during the Phase 2 source audit.

The source explicitly states that:

- the 4,096 inputs connect to 10,000 nonlinear ReLU units;
- input-to-hidden weights are sampled uniformly from (−0.0001, 0.0001);
- the ReLU units connect to one prediction unit;
- hidden-to-output weights are initialized to zero.

It does not state whether either layer has a bias parameter.


## Paper-defined decisions

- Input width: 4,096
- Hidden width: 10,000
- Hidden nonlinearity: ReLU
- Output width: one scalar prediction
- Input-to-hidden initialization: Uniform(−0.0001, 0.0001)
- Hidden-to-output initialization: exactly zero

## Reproduction-protocol decisions

- Model-initialization seed: 20,260,202
- Dtype: float32
- Device: CUDA when available, otherwise CPU
- Bias default: disabled
- Model construction and faithful reset do not advance the global torch RNG

The no-bias default is a minimum-intervention assumption: it avoids adding parameters the
source does not mention. Bias support remains configurable, but enabling it requires an
explicit initialization value so an assumption cannot be introduced silently.

## Open questions

- UNRESOLVED_FROM_SOURCE: whether either linear transformation includes a bias.
- UNRESOLVED_FROM_SOURCE: exact NetworkIDBD state variables and their memory cost.
- UNRESOLVED_FROM_SOURCE: loss scaling, SGD learning rate, training horizon, evaluation
  cadence, and number of runs. These affect Phase 3 and are not decided here.


## Environment and hardware

Capturing the actual runtime matters because this is a 40.97-million-parameter network.
Absolute bitwise reproducibility can still depend on hardware, drivers, CUDA kernels, and
library versions even when deterministic behavior is requested.


In [1]:
from __future__ import annotations

import gc
import math
import os
import platform
import random
import sys
from dataclasses import asdict
from typing import Any

import numpy as np
import pandas as pd
import torch

from src.models import ModelConfig, NoisyMNISTMLP
from src.utils import seed_everything


def gibibytes(num_bytes: int) -> float:
    """Convert bytes to GiB."""

    return num_bytes / 1024**3


def host_memory() -> tuple[int, int]:
    """Return total and currently available host memory from Linux procfs."""

    total_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    available_bytes = 0
    with open("/proc/meminfo", encoding="utf-8") as meminfo:
        for line in meminfo:
            if line.startswith("MemAvailable:"):
                available_bytes = int(line.split()[1]) * 1024
                break
    return total_bytes, available_bytes


total_ram, available_ram = host_memory()
print(f"Python version:      {platform.python_version()}")
print(f"PyTorch version:     {torch.__version__}")
print(f"NumPy version:       {np.__version__}")
print(f"pandas version:      {pd.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:            {torch.cuda.get_device_name(0)}")
    print(
        "GPU memory:          "
        f"{gibibytes(torch.cuda.get_device_properties(0).total_memory):.2f} GiB"
    )
else:
    print("GPU name:            n/a (CPU execution)")
print(f"Logical CPU count:   {os.cpu_count()}")
print(f"Host RAM total:      {gibibytes(total_ram):.2f} GiB")
print(f"Host RAM available:  {gibibytes(available_ram):.2f} GiB at inspection")


Python version:      3.12.14
PyTorch version:     2.13.0+cpu
NumPy version:       2.5.2
pandas version:      3.0.5
CUDA available:      False
GPU name:            n/a (CPU execution)
Logical CPU count:   4
Host RAM total:      6.15 GiB
Host RAM available:  2.25 GiB at inspection


## Configuration

The reusable implementation lives in
[src/models/noisymnist_mlp.py](src/models/noisymnist_mlp.py). Notebook cells orchestrate
verification rather than hiding architecture or initialization choices.


In [2]:
MODEL_INITIALIZATION_SEED = 20_260_202  # REPRODUCTION_PROTOCOL
execution_device = "cuda" if torch.cuda.is_available() else "cpu"

model_config = ModelConfig(
    initialization_seed=MODEL_INITIALIZATION_SEED,
    use_bias=False,  # REPRODUCTION_PROTOCOL: source does not specify biases.
    bias_initial_value=None,
    dtype=torch.float32,
    device=execution_device,
)
try:
    ModelConfig(input_hidden_low=-0.0002, input_hidden_high=0.0002)
except ValueError:
    exact_initialization_contract_pass = True
else:
    exact_initialization_contract_pass = False
assert exact_initialization_contract_pass
seed_everything(MODEL_INITIALIZATION_SEED)

model_config_display = asdict(model_config)
model_config_display["dtype"] = str(model_config.dtype)
display(pd.Series(model_config_display, name="value").to_frame())


,value
input_size,4096
hidden_size,10000
output_size,1
input_hidden_low,-0.0001
input_hidden_high,0.0001
initialization_seed,20260202
use_bias,False
bias_initial_value,None
dtype,torch.float32
device,cpu


## Model implementation

NoisyMNISTMLP contains two explicit torch Linear modules and an explicit ReLU:

4,096 → Linear(4,096, 10,000) → ReLU → Linear(10,000, 1)

It exposes first- and second-layer weights, hidden activations, and the prediction. Optional
biases exist only for an explicitly labeled **ABLATION_OR_EXTENSION**; they are disabled in
the faithful default because the source does not specify bias handling.
Inputs may be one experience shaped [4096] or a diagnostic batch shaped
[batch, 4096]. The faithful learning protocol will nevertheless remain batch size one.


In [3]:
def numpy_states_equal(left: tuple[Any, ...], right: tuple[Any, ...]) -> bool:
    """Compare legacy NumPy global RNG states exactly."""

    return (
        left[0] == right[0]
        and np.array_equal(left[1], right[1])
        and left[2:] == right[2:]
    )


python_rng_before = random.getstate()
numpy_rng_before = np.random.get_state()
torch_cpu_rng_before = torch.get_rng_state().clone()
cuda_rng_before = (
    [state.clone() for state in torch.cuda.get_rng_state_all()]
    if torch.cuda.is_available()
    else []
)

model = NoisyMNISTMLP(model_config)

cuda_rng_after = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []
construction_rng_isolated = bool(
    python_rng_before == random.getstate()
    and numpy_states_equal(numpy_rng_before, np.random.get_state())
    and torch.equal(torch_cpu_rng_before, torch.get_rng_state())
    and len(cuda_rng_before) == len(cuda_rng_after)
    and all(
        torch.equal(before, after)
        for before, after in zip(cuda_rng_before, cuda_rng_after, strict=True)
    )
)
assert construction_rng_isolated
print(model)
print("Model construction and initialization left global RNG states unchanged.")


NoisyMNISTMLP(
  (input_to_hidden): Linear(in_features=4096, out_features=10000, bias=False)
  (hidden_to_output): Linear(in_features=10000, out_features=1, bias=False)
)
Model construction and initialization left global RNG states unchanged.


## Shape and parameter-count verification

With the no-bias protocol default:

- first layer: 10,000 × 4,096 = 40,960,000 weights;
- second layer: 1 × 10,000 = 10,000 weights;
- total: 40,970,000 trainable scalar parameters.


In [4]:
expected_first_layer_shape = (10_000, 4_096)
expected_second_layer_shape = (1, 10_000)
expected_first_layer_parameters = 10_000 * 4_096
expected_second_layer_parameters = 10_000
expected_parameter_count = (
    expected_first_layer_parameters + expected_second_layer_parameters
)

shape_checks_pass = bool(
    tuple(model.first_layer_weights.shape) == expected_first_layer_shape
    and tuple(model.second_layer_weights.shape) == expected_second_layer_shape
)
parameter_count_pass = model.parameter_count() == expected_parameter_count
bias_default_pass = (
    model.first_layer_bias is None and model.second_layer_bias is None
)

assert shape_checks_pass
assert parameter_count_pass
assert bias_default_pass

parameter_shapes = pd.DataFrame(
    [
        {
            "parameter": name,
            "shape": tuple(parameter.shape),
            "numel": parameter.numel(),
            "dtype": str(parameter.dtype),
            "device": str(parameter.device),
        }
        for name, parameter in model.named_parameters()
    ]
)
display(parameter_shapes)
print(f"Exact trainable parameter count: {model.parameter_count():,}")


,parameter,shape,numel,dtype,device
0,input_to_hidden.weight,"(10000, 4096)",40960000,torch.float32,cpu
1,hidden_to_output.weight,"(1, 10000)",10000,torch.float32,cpu


Exact trainable parameter count: 40,970,000


## Initialization verification

The complete 40.96-million-value first layer is inspected, not a sample. The output-layer
zero check is also exact.


In [5]:
with torch.no_grad():
    first_layer_min = float(model.first_layer_weights.min().cpu())
    first_layer_max = float(model.first_layer_weights.max().cpu())
    first_layer_mean = float(model.first_layer_weights.mean().cpu())
    first_layer_std = float(
        model.first_layer_weights.std(unbiased=False).cpu()
    )
    below_count = int(
        torch.count_nonzero(
            model.first_layer_weights < model_config.input_hidden_low
        ).cpu()
    )
    above_count = int(
        torch.count_nonzero(
            model.first_layer_weights > model_config.input_hidden_high
        ).cpu()
    )
    outside_count = below_count + above_count
    outside_proportion = outside_count / model.first_layer_weights.numel()
    output_nonzero_count = int(
        torch.count_nonzero(model.second_layer_weights).cpu()
    )

expected_uniform_mean = (
    model_config.input_hidden_low + model_config.input_hidden_high
) / 2.0
expected_uniform_std = (
    model_config.input_hidden_high - model_config.input_hidden_low
) / math.sqrt(12.0)

initialization_bounds_pass = bool(
    first_layer_min >= model_config.input_hidden_low
    and first_layer_max <= model_config.input_hidden_high
    and outside_count == 0
)
output_zero_pass = output_nonzero_count == 0

assert initialization_bounds_pass
assert output_zero_pass

initialization_statistics = pd.DataFrame(
    {
        "observed": [
            first_layer_min,
            first_layer_max,
            first_layer_mean,
            first_layer_std,
            outside_proportion,
            float(output_nonzero_count),
        ],
        "reference": [
            model_config.input_hidden_low,
            model_config.input_hidden_high,
            expected_uniform_mean,
            expected_uniform_std,
            0.0,
            0.0,
        ],
    },
    index=[
        "first-layer minimum",
        "first-layer maximum",
        "first-layer mean",
        "first-layer std (population)",
        "proportion outside required range",
        "nonzero output-layer weights",
    ],
)
display(initialization_statistics)


,observed,reference
first-layer minimum,-1.000000e-04,-0.000100
first-layer maximum,9.999998e-05,0.000100
first-layer mean,-7.974048e-09,0.000000
first-layer std (population),5.773939e-05,0.000058
proportion outside required range,0.000000e+00,0.000000
nonzero output-layer weights,0.000000e+00,0.000000


## Deterministic initialization tests

One intentional parameter snapshot is used temporarily to test exact equality without
holding multiple giant model objects. It is deleted immediately after verification.


In [6]:
reference_state = {
    name: parameter.detach().clone()
    for name, parameter in model.named_parameters()
}

model.reset_parameters_from_config(MODEL_INITIALIZATION_SEED)
same_seed_bit_identical_pass = all(
    torch.equal(reference_state[name], parameter)
    for name, parameter in model.named_parameters()
)

model.reset_parameters_from_config(MODEL_INITIALIZATION_SEED + 1)
different_seed_changes_first_layer_pass = not torch.equal(
    reference_state["input_to_hidden.weight"],
    model.first_layer_weights,
)
different_seed_output_still_zero_pass = (
    int(torch.count_nonzero(model.second_layer_weights).cpu()) == 0
)

model.reset_parameters_from_config(MODEL_INITIALIZATION_SEED)
constructor_state_restored_pass = all(
    torch.equal(reference_state[name], parameter)
    for name, parameter in model.named_parameters()
)

assert same_seed_bit_identical_pass
assert different_seed_changes_first_layer_pass
assert different_seed_output_still_zero_pass
assert constructor_state_restored_pass

del reference_state
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

deterministic_results = pd.Series(
    {
        "same seed gives bit-identical parameters": same_seed_bit_identical_pass,
        "different seed changes input-hidden weights": different_seed_changes_first_layer_pass,
        "different seed leaves output weights zero": different_seed_output_still_zero_pass,
        "constructor-seed state restored": constructor_state_restored_pass,
        "global RNG state isolated": construction_rng_isolated,
    },
    name="passed",
)
display(deterministic_results.to_frame())


,passed
same seed gives bit-identical parameters,True
different seed changes input-hidden weights,True
different seed leaves output weights zero,True
constructor-seed state restored,True
global RNG state isolated,True


## Forward-pass tests

Since the hidden-to-output weights are initially zero, the initial prediction must be
exactly zero for every finite input. Hidden activations need not be zero.


In [7]:
single_input = torch.linspace(
    0.0,
    1.0,
    steps=model_config.input_size,
    dtype=model_config.dtype,
    device=model_config.device,
)
batched_input = single_input.unsqueeze(0)

with torch.no_grad():
    single_result = model.forward_with_activations(single_input)
    batched_result = model.forward_with_activations(batched_input)
    direct_prediction = model(single_input)

forward_shape_pass = bool(
    single_result.hidden_activations.shape == (10_000,)
    and single_result.prediction.shape == ()
    and batched_result.hidden_activations.shape == (1, 10_000)
    and batched_result.prediction.shape == (1,)
    and direct_prediction.shape == ()
)
forward_finite_pass = bool(
    torch.isfinite(single_result.hidden_activations).all()
    and torch.isfinite(single_result.prediction)
    and torch.isfinite(batched_result.hidden_activations).all()
    and torch.isfinite(batched_result.prediction).all()
)
forward_dtype_device_pass = bool(
    single_result.hidden_activations.dtype == model_config.dtype
    and single_result.prediction.dtype == model_config.dtype
    and single_result.hidden_activations.device == model.first_layer_weights.device
    and single_result.prediction.device == model.first_layer_weights.device
)
initial_prediction_zero_pass = bool(
    float(single_result.prediction.cpu()) == 0.0
    and float(batched_result.prediction.item()) == 0.0
    and torch.equal(direct_prediction, single_result.prediction)
)

assert forward_shape_pass
assert forward_finite_pass
assert forward_dtype_device_pass
assert initial_prediction_zero_pass

print(f"Single input shape:       {tuple(single_input.shape)}")
print(f"Single hidden shape:      {tuple(single_result.hidden_activations.shape)}")
print(f"Single prediction shape:  {tuple(single_result.prediction.shape)}")
print(f"Batch input shape:        {tuple(batched_input.shape)}")
print(f"Batch hidden shape:       {tuple(batched_result.hidden_activations.shape)}")
print(f"Batch prediction shape:   {tuple(batched_result.prediction.shape)}")
print(f"Initial prediction:       {float(single_result.prediction.cpu()):.1f}")
print(
    "Active hidden fraction:  "
    f"{float((single_result.hidden_activations > 0).float().mean().cpu()):.6f}"
)


Single input shape:       (4096,)
Single hidden shape:      (10000,)
Single prediction shape:  ()
Batch input shape:        (1, 4096)
Batch hidden shape:       (1, 10000)
Batch prediction shape:   (1,)
Initial prediction:       0.0
Active hidden fraction:  0.499600


## Memory accounting

Float32 uses four bytes per scalar. Momentum-free torch.optim.SGD has no persistent
optimizer-state tensor; gradients still require parameter-sized storage. This table is
analytical and excludes allocator, module, autograd-graph, input, and framework overhead.

NetworkIDBD memory is deliberately not finalized because the official source does not
publish its equations or exact state structure. If a future verified derivation requires
one full-size state tensor per parameter, each such tensor would add another parameter-sized
allocation. This observation is capacity planning, not an assumed algorithm.


In [8]:
parameter_bytes = model.parameter_bytes()
gradient_bytes = parameter_bytes
momentum_free_sgd_state_bytes = 0
one_full_parameter_state_bytes = parameter_bytes

memory_accounting = pd.DataFrame(
    [
        {
            "component": "parameters",
            "bytes": parameter_bytes,
            "MiB": parameter_bytes / 1024**2,
            "status": "exact",
        },
        {
            "component": "gradients after backward",
            "bytes": gradient_bytes,
            "MiB": gradient_bytes / 1024**2,
            "status": "exact parameter-sized storage",
        },
        {
            "component": "momentum-free SGD persistent state",
            "bytes": momentum_free_sgd_state_bytes,
            "MiB": 0.0,
            "status": "exact",
        },
        {
            "component": "each hypothetical full-size adaptive-state tensor",
            "bytes": one_full_parameter_state_bytes,
            "MiB": one_full_parameter_state_bytes / 1024**2,
            "status": "capacity illustration only",
        },
    ]
)
display(memory_accounting)

base_training_storage_bytes = parameter_bytes + gradient_bytes
print(f"Parameters:                    {parameter_bytes / 1024**2:.2f} MiB")
print(f"Gradients:                     {gradient_bytes / 1024**2:.2f} MiB")
print("Momentum-free SGD state:       0.00 MiB")
print(
    "Parameters + gradients:        "
    f"{base_training_storage_bytes / 1024**2:.2f} MiB minimum"
)
print(
    "Each full parameter-sized state tensor would add: "
    f"{one_full_parameter_state_bytes / 1024**2:.2f} MiB"
)
print("NetworkIDBD exact state:       UNRESOLVED_FROM_SOURCE")

memory_accounting_pass = bool(
    parameter_bytes == expected_parameter_count * 4
    and gradient_bytes == parameter_bytes
    and momentum_free_sgd_state_bytes == 0
)
assert memory_accounting_pass


,component,bytes,MiB,status
0,parameters,163880000,156.288147,exact
1,gradients after backward,163880000,156.288147,exact parameter-sized storage
2,momentum-free SGD persistent state,0,0.000000,exact
3,each hypothetical full-size adaptive-state tensor,163880000,156.288147,capacity illustration only


Parameters:                    156.29 MiB
Gradients:                     156.29 MiB
Momentum-free SGD state:       0.00 MiB
Parameters + gradients:        312.58 MiB minimum
Each full parameter-sized state tensor would add: 156.29 MiB
NetworkIDBD exact state:       UNRESOLVED_FROM_SOURCE


## Verification checklist

Checklist values below are computed from runtime assertions rather than hard-coded.


In [9]:
verification_checks = [
    ("faithful initialization interval is configuration-guarded", exact_initialization_contract_pass),
    ("exact parameter shapes", shape_checks_pass),
    ("exact parameter count", parameter_count_pass),
    ("bias default explicitly disabled", bias_default_pass),
    ("input-hidden weights inside required range", initialization_bounds_pass),
    ("hidden-output weights exactly zero", output_zero_pass),
    ("same-seed initialization bit-identical", same_seed_bit_identical_pass),
    ("different seed changes input-hidden weights", different_seed_changes_first_layer_pass),
    ("forward shapes correct", forward_shape_pass),
    ("activations and predictions finite", forward_finite_pass),
    ("dtype and device correct", forward_dtype_device_pass),
    ("initial scalar prediction exactly zero", initial_prediction_zero_pass),
    ("global RNG state isolated", construction_rng_isolated),
    ("memory accounting internally consistent", memory_accounting_pass),
]

for check_name, passed in verification_checks:
    print(f"- [{'PASS' if passed else 'FAIL'}] {check_name}")
assert all(passed for _, passed in verification_checks)


- [PASS] faithful initialization interval is configuration-guarded
- [PASS] exact parameter shapes
- [PASS] exact parameter count
- [PASS] bias default explicitly disabled
- [PASS] input-hidden weights inside required range
- [PASS] hidden-output weights exactly zero
- [PASS] same-seed initialization bit-identical
- [PASS] different seed changes input-hidden weights
- [PASS] forward shapes correct
- [PASS] activations and predictions finite
- [PASS] dtype and device correct
- [PASS] initial scalar prediction exactly zero
- [PASS] global RNG state isolated
- [PASS] memory accounting internally consistent


## Results

The exact 40.97-million-parameter model was instantiated and tested on the available
device. Complete-layer initialization statistics, deterministic-reset tests, forward
tests, and analytical memory accounting passed. No optimization or training occurred.

## Deviations from paper

- Biases are disabled as an explicitly labeled minimum-intervention protocol decision
  because the source is silent.
- Float32, device selection, and the initialization seed are reproduction choices.
- No architectural dimensions or paper-defined weight initializations were changed.

## Next phase

Phase 3 may implement the batch-size-one SGD mechanism. Before claiming a faithful training
result, it must mark the NoisyMNIST loss scaling, learning rate, training horizon,
evaluation cadence, and run count as UNRESOLVED_FROM_SOURCE unless stronger primary
material is found. NetworkIDBD must not be implemented before a separate source-verified
mathematical derivation.
